In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd

from src.generator import generate_asset_register
from src.models import PriorityTier, ScoredAsset
from src.scoring import score_all

In [2]:
assets = generate_asset_register()

# test a high risk asset
target = next(a for a in assets
              if a.asset_type.value == 'Transformer' and a.age_ratio > 0.85)
print("target asset:", target)

# === calculate PoF manually ===
age_factor       = min(target.age_ratio, 1.0)
condition_factor = (10 - target.inspection_score) / 9
fault_factor     = min(target.fault_count_5yr / 5, 1.0)

pof = (0.5 * age_factor +
       0.3 * condition_factor +
       0.2 * fault_factor)

print(f"\nPoF calculation:")
print(f"  age     = {age_factor:.3f}  (weight 50%)")
print(f"  condition     = {condition_factor:.3f}  (weight 30%)")
print(f"  fault history = {fault_factor:.3f}  (weight 20%)")
print(f"  PoF score      = {pof:.4f}")

# === calculate CoF manually ===
customer_factor  = np.log1p(target.customers_affected_if_fail) / np.log1p(2000)
critical_factor  = 1.0 if target.is_critical_node else 0.3
cost_factor      = np.log1p(target.replacement_cost_nzd) / np.log1p(500_000)

cof = (0.5 * customer_factor +
       0.3 * critical_factor +
       0.2 * cost_factor)

print(f"\nCoF calculation:")
print(f"  customers     = {customer_factor:.3f}  (weight 50%)")
print(f"  critical node     = {critical_factor:.3f}  (weight 30%)")
print(f"  replacement cost = {cost_factor:.3f}  (weight 20%)")
print(f"  CoF score      = {cof:.4f}")

risk = pof * cof
print(f"\nFinal Risk Score = PoF × CoF = {pof:.4f} × {cof:.4f} = {risk:.4f}")

target asset: Asset(asset_id='CE-0302', asset_type=<AssetType.TRANSFORMER: 'Transformer'>, install_year=1996, age_years=30, age_ratio=0.857, zone='Tuakau', voltage_kv=33, customers_affected_if_fail=87, last_inspection_year=2022, inspection_score=1, fault_count_5yr=2, is_critical_node=True, replacement_cost_nzd=249827)

PoF calculation:
  age     = 0.857  (weight 50%)
  condition     = 1.000  (weight 30%)
  fault history = 0.400  (weight 20%)
  PoF score      = 0.8085

CoF calculation:
  customers     = 0.589  (weight 50%)
  critical node     = 1.000  (weight 30%)
  replacement cost = 0.947  (weight 20%)
  CoF score      = 0.7839

Final Risk Score = PoF × CoF = 0.8085 × 0.7839 = 0.6338


In [3]:
scored = score_all(assets)

# Change to DataFrame
rows = []
for s in scored:
    rows.append({
        'asset_id':   s.asset.asset_id,
        'asset_type': s.asset.asset_type.value,
        'zone':       s.asset.zone,
        'age_years':  s.asset.age_years,
        'age_ratio':  s.asset.age_ratio,
        'pof_score':  s.pof_score,
        'cof_score':  s.cof_score,
        'risk_score': s.risk_score,
        'priority':   s.priority_tier.value,
        'customers':  s.asset.customers_affected_if_fail,
        'cost_nzd':   s.asset.replacement_cost_nzd,
    })
df_scored = pd.DataFrame(rows)

print("Priority Distribution:")
print(df_scored['priority'].value_counts())

#  Annual budget allocation
BUDGET = 5_000_000
p1 = df_scored[df_scored['priority'] == 'P1_Critical']\
     .sort_values('risk_score', ascending=False).copy()
p1['cumulative_cost'] = p1['cost_nzd'].cumsum()
p1['within_budget']   = p1['cumulative_cost'] <= BUDGET

print(f"\nP1 Assets: {len(p1)}")
print(f"P1 Total Replacement Cost: NZD {p1['cost_nzd'].sum():,.0f}")
print(f"NZD 5M Budget Within: {p1['within_budget'].sum()} P1 Assets")

df_scored.to_csv('../data/processed/asset_scores.csv', index=False)
print("\nSaved to data/processed/asset_scores.csv")

Priority Distribution:
priority
P3_Medium      189
P2_High        178
P4_Low          76
P1_Critical     57
Name: count, dtype: int64

P1 Assets: 57
P1 Total Replacement Cost: NZD 15,271,072
NZD 5M Budget Within: 16 P1 Assets

Saved to data/processed/asset_scores.csv
